In [1]:
!pip install pdfplumber -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.6 MB/s eta 0:00:00


In [2]:
import pdfplumber
import pandas as pd

rows = []

with pdfplumber.open("annexure-1-state-wise-handbooks-of-heat-risk.pdf") as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()
        for table in tables:
            for row in table:
                if row is None:
                    continue
                # Clean each cell
                row = [str(c).strip() if c else '' for c in row]
                # We need rows with 6 columns and a numeric score
                if len(row) >= 5:
                    try:
                        score = float(row[2]) if row[2] else float(row[3])
                        rows.append(row)
                    except ValueError:
                        continue

df_raw = pd.DataFrame(rows)
print(f"Raw rows extracted: {len(df_raw)}")
print(df_raw.head(20))

Raw rows extracted: 724
              0                          1         2  3         4   5
0   WEST BENGAL  North Twenty Four\nPargan  0.809549  4      High   1
1   WEST BENGAL  South Twenty Four\nPargan  0.805079  4      High   2
2   WEST BENGAL                    Kolkata  0.796712  4      High   3
3   WEST BENGAL                   Puruliya  0.785942  4      High   4
4   WEST BENGAL            Purba Medinipur  0.783574  4      High   5
5   WEST BENGAL                    Bankura  0.782579  4      High   6
6   WEST BENGAL                    Hooghly  0.769255  3  moderate   7
7   WEST BENGAL             Medinipur West    0.7663  3  moderate   8
8   WEST BENGAL          Paschim Bardhaman  0.764875  3  moderate   9
9   WEST BENGAL                     Howrah  0.764157  3  moderate  10
10  WEST BENGAL                      Nadia  0.754871  3  moderate  11
11  WEST BENGAL                    Birbhum  0.746812  3  moderate  12
12  WEST BENGAL            Purba Bardhaman  0.742799  3  moderate 

In [3]:
df_hri = df_raw.copy()
df_hri.columns = ['state', 'district', 'hri_normalised',
                   'hri_score', 'hri_category', 'intrastate_rank']

# Fix newlines in district names
df_hri['district'] = df_hri['district'].str.replace('\n', ' ', regex=False)

# Fix types
df_hri['hri_normalised'] = pd.to_numeric(df_hri['hri_normalised'], errors='coerce')
df_hri['hri_score'] = pd.to_numeric(df_hri['hri_score'], errors='coerce')
df_hri['intrastate_rank'] = pd.to_numeric(df_hri['intrastate_rank'], errors='coerce')

# Standardise case for joining later
df_hri['state'] = df_hri['state'].str.strip().str.title()
df_hri['district'] = df_hri['district'].str.strip().str.title()

# Sanity check
print(df_hri.dtypes)
print(f"\nTotal districts: {len(df_hri)}")
print(f"States covered: {df_hri['state'].nunique()}")
print(f"\nHRI score distribution:")
print(df_hri['hri_category'].value_counts())
print(f"\nScore range: {df_hri['hri_normalised'].min():.3f} to {df_hri['hri_normalised'].max():.3f}")

# Save
df_hri.to_csv("ceew_district_hri.csv", index=False)
print("\nSaved: ceew_district_hri.csv")

state               object
district            object
hri_normalised     float64
hri_score            int64
hri_category        object
intrastate_rank      int64
dtype: object

Total districts: 724
States covered: 34

HRI score distribution:
hri_category
High         257
moderate     208
Very high    146
low           77
very low      36
Name: count, dtype: int64

Score range: 0.000 to 1.000

Saved: ceew_district_hri.csv


In [4]:
import requests
import pandas as pd
from io import StringIO

# List of state codes from pratapvardhan repo
states = ['AN','AP','AS','BR','DD','GA','GJ','HP','JK','KA','KL','LH','MH','ML','MN','MZ','NL','SK','TG','TR','WB']

base_url = "https://raw.githubusercontent.com/pratapvardhan/NFHS-5/master/district-level/NFHS-5-{code}-{name}.csv"

# State code to name mapping
state_names = {
    'AN': 'Andaman-and-Nicobar-Island',
    'AP': 'Andhra-Pradesh',
    'AS': 'Assam',
    'BR': 'Bihar',
    'DD': 'Dadra-Nagar-Haveli-and-Daman-Diu',
    'GA': 'Goa',
    'GJ': 'Gujarat',
    'HP': 'Himachal-Pradesh',
    'JK': 'Jammu-and-Kashmir',
    'KA': 'Karnataka',
    'KL': 'Kerala',
    'LH': 'Ladakh',
    'MH': 'Maharashtra',
    'ML': 'Meghalaya',
    'MN': 'Manipur',
    'MZ': 'Mizoram',
    'NL': 'Nagaland',
    'SK': 'Sikkim',
    'TG': 'Telangana',
    'TR': 'Tripura',
    'WB': 'West-Bengal'
}

dfs = []
for code, name in state_names.items():
    url = f"https://raw.githubusercontent.com/pratapvardhan/NFHS-5/master/district-level/NFHS-5-{code}-{name}.csv"
    try:
        r = requests.get(url)
        if r.status_code == 200:
            df = pd.read_csv(StringIO(r.text))
            dfs.append(df)
            print(f"✓ {name}: {df['District'].nunique()} districts")
        else:
            print(f"✗ {name}: HTTP {r.status_code}")
    except Exception as e:
        print(f"✗ {name}: {e}")

nfhs = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(nfhs)}")
print(f"Total districts: {nfhs['District'].nunique()}")
print(f"Total states: {nfhs['State'].nunique()}")

# Find indicators
print("\n--- LBW matches ---")
print(nfhs[nfhs['Indicator'].str.contains(
    'low birth weight', case=False)]['Indicator'].unique())

print("\n--- Distance matches ---")
print(nfhs[nfhs['Indicator'].str.contains(
    'distance', case=False)]['Indicator'].unique())

✓ Andaman-and-Nicobar-Island: 3 districts
✓ Andhra-Pradesh: 13 districts
✓ Assam: 33 districts
✓ Bihar: 38 districts
✓ Dadra-Nagar-Haveli-and-Daman-Diu: 3 districts
✓ Goa: 2 districts
✓ Gujarat: 33 districts
✓ Himachal-Pradesh: 12 districts
✓ Jammu-and-Kashmir: 20 districts
✓ Karnataka: 30 districts
✓ Kerala: 14 districts
✓ Ladakh: 2 districts
✓ Maharashtra: 36 districts
✓ Meghalaya: 11 districts
✓ Manipur: 9 districts
✓ Mizoram: 8 districts
✓ Nagaland: 11 districts
✓ Sikkim: 4 districts
✓ Telangana: 31 districts
✓ Tripura: 8 districts
✓ West-Bengal: 20 districts

Total rows: 35464
Total districts: 340
Total states: 21

--- LBW matches ---
[]

--- Distance matches ---
[]


In [5]:
# See every unique indicator
all_indicators = nfhs['Indicator'].unique()
print(f"Total indicators: {len(all_indicators)}")
print("\nAll indicators:")
for i, ind in enumerate(all_indicators):
    print(f"{i}: {ind}")

Total indicators: 104

All indicators:
0: 1. Female population age 6 years and above who ever attended school (%)
1: 2. Population below age 15 years (%)
2: 3. Sex ratio of the total population (females per 1,000 males)
3: 4. Sex ratio at birth for children born in the last five years (females per 1,000 males)
4: 5. Children under age 5 years whose birth was registered with the civil authority (%)
5: 6. Deaths in the last 3 years registered with the civil authority (%)
6: 7. Population living in households with electricity (%)
7: 8. Population living in households with an improved drinking-water source1 (%)
8: 9. Population living in households that use an improved sanitation facility2 (%)
9: 10. Households using clean fuel for cooking3 (%)
10: 11. Households using iodized salt (%)
11: 12. Households with any usual member covered under a health insurance/financing scheme (%)
12: 13. Children age 5 years who attended pre-primary school during the school year 2019-20 (%)
13: 14. Women wh

In [6]:
indicators = {
    'underweight_pct': '76. Children under 5 years who are underweight (weight-for-age)18 (%)',
    'anaemia_pregnant_pct': '83. Pregnant women age 15-49 years who are anaemic (<11.0 g/dl)22 (%)',
    'institutional_births_pct': '42. Institutional births (%)',
    'antenatal_first_trimester_pct': '32. Mothers who had an antenatal check-up in the first trimester (%)',
    'postnatal_care_pct': '38. Mothers who received postnatal care from a doctor/nurse/LHV/ANM/midwife/other health personnel within 2 days of delivery (%)'
}

dfs_out = []
for col_name, indicator_str in indicators.items():
    tmp = nfhs[nfhs['Indicator'] == indicator_str][
        ['State', 'District', 'NFHS-5']
    ].copy()
    tmp.columns = ['state', 'district', 'value']
    tmp['metric'] = col_name
    tmp['value'] = pd.to_numeric(tmp['value'], errors='coerce')
    dfs_out.append(tmp)

nfhs_wide = pd.concat(dfs_out)
nfhs_pivot = nfhs_wide.pivot_table(
    index=['state', 'district'],
    columns='metric',
    values='value'
).reset_index()

nfhs_pivot.columns.name = None
nfhs_pivot['state'] = nfhs_pivot['state'].str.strip().str.title()
nfhs_pivot['district'] = nfhs_pivot['district'].str.strip().str.title()

print(f"NFHS districts: {len(nfhs_pivot)}")
print(nfhs_pivot.head(10))

# Now merge with CEEW HRI
merged = pd.merge(nfhs_pivot, df_hri, on=['state', 'district'], how='inner')
print(f"\nMatched districts: {len(merged)}")
print(f"Unmatched: {len(nfhs_pivot) - len(merged)}")

merged.to_csv("scatter_data.csv", index=False)
print("\nSaved: scatter_data.csv")
print(merged[['state','district','underweight_pct',
              'anaemia_pregnant_pct','institutional_births_pct',
              'hri_normalised','hri_category']].head(15))

NFHS districts: 341
                        state                district  anaemia_pregnant_pct  \
0  Andaman And Nicobar Island                 Nicobar                   NaN   
1  Andaman And Nicobar Island  North & Middle Andaman                   NaN   
2  Andaman And Nicobar Island           South Andaman                   NaN   
3              Andhra Pradesh               Anantapur                  54.9   
4              Andhra Pradesh                Chittoor                  52.7   
5              Andhra Pradesh           East Godavari                   NaN   
6              Andhra Pradesh                  Guntur                  51.9   
7              Andhra Pradesh                 Krishna                   NaN   
8              Andhra Pradesh                 Kurnool                  67.8   
9              Andhra Pradesh                Prakasam                   NaN   

   antenatal_first_trimester_pct  institutional_births_pct  \
0                           62.8                